# Wildfire Prediction Training on Google Colab

This notebook trains the XGBoost and CNN models. Run once completely, and save the resulting information before running the rmd file to generate the final html render.


In [ ]:
codename <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
repo_url <- paste0("https://packagemanager.posit.co/cran/__linux__/", codename, "/latest")
options(repos = c(CRAN = repo_url))

message(paste("Using fast binary repository:", repo_url))

install_packages <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    message(paste("Installing fast binary:", pkg))
    install.packages(pkg)
  }
}
pkgs <- c("tidyverse", "ranger", "xgboost", "caret", "pROC", "PRROC", "Matrix", "nnet", "lubridate", "gridExtra", "torch", "luz", "viridis", "coro")
sapply(pkgs, install_packages)

library(tidyverse)
library(ranger)
library(xgboost)
library(caret)
library(pROC)
library(PRROC)
library(Matrix)
library(nnet)
library(lubridate)
library(gridExtra)
library(torch)
library(luz)
library(viridis)
library(coro)

if (cuda_is_available()) {
  message("GPU is available for Torch: ", cuda_get_device_name(0))
  device <- "cuda"
} else {
  message("GPU not found for Torch. Using CPU.")
  device <- "cpu"
}

message("Packages loaded successfully!")

In [ ]:
file.info("/content/final_dataset.rds")$size

In [ ]:
data_path <- "/content/final_dataset.rds"

if (!file.exists(data_path)) {
  message("File not found in /content. Please upload 'final_dataset.rds' now.")
  system("python3 -c 'from google.colab import files; files.upload()'" )
}

if (file.exists(data_path)) {
  df <- readRDS(data_path)
  message(paste("Dataset successfully loaded! Rows:", nrow(df)))
} else {
  stop("Upload failed or file named incorrectly. Please ensure the file is named 'final_dataset.rds'.")
}

In [ ]:
model_df <- df %>%
  arrange(grid_id, month_date) %>%
  group_by(grid_id) %>%
  mutate(
    y_next = as.factor(dplyr::lead(fire_occurred, 1)),
    fire_roll12 = as.numeric(stats::filter(fire_occurred, rep(1, 12), sides = 1)),
    lag_fire_1m = lag(fire_occurred, 1, default = 0),
    lat_bin = cut(lat, breaks = 15, labels = FALSE),
    lon_bin = cut(lon, breaks = 15, labels = FALSE)
  ) %>%
  ungroup() %>%
  na.omit()

features_consolidated <- c(
  "lat", "lon", "fire_occurred", "n_fires", "total_acres", "ppt", "tmean", "vpd",
  "soil_temp_l1", "soil_water_l1", "soil_water_l2", "evaporation", "u_wind", "v_wind",
  "precip", "month_sin", "month_cos", "fire_count_this_month",
  "rolling_fire_3m_inclusive", "rolling_fire_count_3m",
  "rolling_fire_12m_inclusive", "rolling_fire_count_12m",
  "rolling_acres_3m_inclusive", "rolling_mean_size_3m",
  "lag_ppt_1m", "lag_ppt_2m", "lag_vpdmax_1m", "lag_vpdmax_2m",
  "lag_tmean_1m", "lag_tmean_2m", "lag_soil_temp_1m", "lag_soil_temp_2m",
  "lag_soil_water_1m", "lag_soil_water_2m",
  "lag_deep_water_1m", "lag_deep_water_2m", "lag_evap_1m", "lag_evap_2m",
  "lag_u_wind_1m", "lag_u_wind_2m", "lag_v_wind_1m", "lag_v_wind_2m",
  "lag_precip_grib_1m", "lag_precip_grib_2m",
  "fire_roll12", "lag_fire_1m", "lat_bin", "lon_bin"
)

missing_cols <- setdiff(features_consolidated, names(model_df))
if(length(missing_cols) > 0) {
  warning("Missing columns: ", paste(missing_cols, collapse=", "))
  features_consolidated <- intersect(features_consolidated, names(model_df))
}

target_var <- "y_next"

model_df$y_next_n <- as.numeric(model_df$y_next) - 1

message("Features Selected: ", length(features_consolidated))
message("Model DataFrame Rows: ", nrow(model_df))

## 4. Expanding Window Validation

In [ ]:
years_sorted <- sort(unique(model_df$year))
min_train_years <- 3

train_ends <- years_sorted[min_train_years:(length(years_sorted) - 1)]
valid_years <- years_sorted[(min_train_years + 1):length(years_sorted)]

f_formula <- as.formula(paste(target_var, "~", paste(features_consolidated, collapse = " + ")))

fold_results <- tibble(
  train_end_year = train_ends,
  valid_year = valid_years
)

nn_weighted_bce_loss <- nn_module(
  "nn_weighted_bce_loss",
  initialize = function(pos_weight) {
    self$pos_weight <- torch_tensor(pos_weight, device = "cuda")
  },
  forward = function(input, target) {
    nnf_binary_cross_entropy_with_logits(input, target, pos_weight = self$pos_weight)
  }
)

FireNN <- nn_module(
  "FireNN",
  initialize = function(in_features) {
    self$fc1 <- nn_linear(in_features, 128)
    self$bn1 <- nn_batch_norm1d(128)
    self$fc2 <- nn_linear(128, 64)
    self$bn2 <- nn_batch_norm1d(64)
    self$fc3 <- nn_linear(64, 32)
    self$output <- nn_linear(32, 1)
    self$dropout <- nn_dropout(0.3)
  },
  forward = function(x) {
    x %>%
      self$fc1() %>% self$bn1() %>% nnf_relu() %>% self$dropout() %>%
      self$fc2() %>% self$bn2() %>% nnf_relu() %>% self$dropout() %>%
      self$fc3() %>% nnf_relu() %>%
      self$output()
  }
)

message(sprintf("Starting loop: %d folds (Test Years: %d - %d)",
                length(train_ends), min(valid_years), max(valid_years)))

run_fold <- function(tr_end, va_year) {
  message(sprintf("\n=== Fold: Train 1992-%d | Test %d ===", tr_end, va_year))

  train_data <- model_df %>% filter(year <= tr_end)
  valid_data <- model_df %>% filter(year == va_year)

  n_pos <- sum(train_data$y_next_n == 1)
  n_neg <- sum(train_data$y_next_n == 0)
  scale_pos_weight <- n_neg / n_pos

  message(sprintf("   Class Imbalance: %d Pos / %d Neg | Scale Weight: %.2f", n_pos, n_neg, scale_pos_weight))

  weights_vec <- ifelse(train_data$y_next_n == 1, scale_pos_weight, 1)
  glm_fit <- glm(f_formula, data = train_data, family = binomial, weights = weights_vec)

  coefs <- coef(glm_fit)
  sorted_coefs <- sort(abs(coefs), decreasing = TRUE)
  top_n <- min(5, length(coefs))
  top_vars <- names(sorted_coefs)[1:top_n]

  message("\n--- Logistic Regression Equation (Top 5 Influential Features) ---")
  message("P(Fire=1) = 1 / (1 + exp(- (Intercept + ...)))")
  message("Log-Odds Equation (Simplified):")

  eq_str <- sprintf("%.4f (Intercept)", coefs["(Intercept)"])
  for(var in top_vars) {
    if(var != "(Intercept)") {
      val <- coefs[var]
      sign_str <- ifelse(val >= 0, "+", "-")
      eq_str <- paste0(eq_str, sprintf(" %s %.4f * %s", sign_str, abs(val), var))
    }
  }
  message(paste0(eq_str, " ... [truncated]"))
  message("-----------------------------------------------------------------\n")

  p_glm <- predict(glm_fit, newdata = valid_data, type = "response")

  auc_glm <- as.numeric(auc(roc(valid_data[[target_var]], p_glm, quiet = TRUE)))

  fg <- p_glm[valid_data$y_next_n == 1]
  bg <- p_glm[valid_data$y_next_n == 0]
  pr_glm <- tryCatch(pr.curve(scores.class0 = fg, scores.class1 = bg)$auc.integral, error = function(e) 0)

  dtrain <- xgb.DMatrix(data = as.matrix(train_data[, features_consolidated]), label = train_data$y_next_n)
dtest  <- xgb.DMatrix(data = as.matrix(valid_data[, features_consolidated]), label = valid_data$y_next_n)

  xgb_params <- list(
    objective = "binary:logistic",
    eval_metric = "aucpr",
    tree_method = "hist",
    device = "cuda",
    eta = 0.05,
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight
  )

  xgb_fit <- xgb.train(
    params = xgb_params,
    data = dtrain,
    nrounds = 500,
    evals = list(train = dtrain, test = dtest),
    early_stopping_rounds = 30,
    verbose = 0
  )

  p_xgb <- predict(xgb_fit, newdata = as.matrix(valid_data[, features_consolidated]))
  auc_xgb <- as.numeric(auc(roc(valid_data[[target_var]], p_xgb, quiet = TRUE)))

  fg_xgb <- p_xgb[valid_data$y_next_n == 1]
  bg_xgb <- p_xgb[valid_data$y_next_n == 0]
  pr_xgb <- tryCatch(pr.curve(scores.class0 = fg_xgb, scores.class1 = bg_xgb)$auc.integral, error = function(e) 0)

  pre_proc <- preProcess(
    train_data[, features_consolidated],
    method = c("nzv", "center", "scale")
  )

  x_train_df <- predict(pre_proc, train_data[, features_consolidated])
  x_valid_df <- predict(pre_proc, valid_data[, features_consolidated])

  valid_features <- names(x_train_df)
  in_features_dim <- length(valid_features)

  x_train <- as.matrix(x_train_df)
  y_train <- matrix(train_data$y_next_n, ncol=1)

  x_valid <- as.matrix(x_valid_df)
  y_valid <- matrix(valid_data$y_next_n, ncol=1)

  train_ds <- tensor_dataset(torch_tensor(x_train, dtype=torch_float()), torch_tensor(y_train, dtype=torch_float()))
  valid_dl <- dataloader(valid_ds, batch_size = 4096, shuffle = FALSE, num_workers = 0)

  fitted <- FireNN %>%
    setup(
      loss = nn_bce_with_logits_loss(pos_weight = torch_tensor(scale_pos_weight, device="cuda")),
      optimizer = optim_adam,
      metrics = list(luz_metric_binary_accuracy_with_logits())
    ) %>%
    set_hparams(in_features = in_features_dim) %>%
    set_opt_hparams(lr = 0.001) %>%
    fit(
      train_ds,
      epochs = 15,
      valid_data = valid_dl,
      verbose = FALSE,
      callbacks = list(luz_callback_early_stopping(patience = 4))
    )

  preds <- predict(fitted, valid_dl)
  p_nn_logits <- as.numeric(as.array(preds))
  p_nn <- 1 / (1 + exp(-p_nn_logits))

  auc_nn <- as.numeric(auc(roc(valid_data[[target_var]], p_nn, quiet = TRUE)))

  fg_nn <- p_nn[valid_data$y_next_n == 1]
  bg_nn <- p_nn[valid_data$y_next_n == 0]
  pr_nn <- tryCatch(pr.curve(scores.class0 = fg_nn, scores.class1 = bg_nn)$auc.integral, error = function(e) 0)

  message(sprintf("   ROC-AUC -> GLM: %.3f | XGB: %.3f | NN: %.3f", auc_glm, auc_xgb, auc_nn))
  message(sprintf("   PR-AUC  -> GLM: %.3f | XGB: %.3f | NN: %.3f", pr_glm, pr_xgb, pr_nn))

  return(list(
    auc_glm = auc_glm, auc_xgb = auc_xgb, auc_nn = auc_nn,
    pr_glm = pr_glm, pr_xgb = pr_xgb, pr_nn = pr_nn
  ))
}

## 5. Execution & Visualization

In [ ]:
results_df <- map_dfr(seq_along(train_ends), function(i) {
  run_fold(train_ends[i], valid_years[i])
})

results_df$year <- valid_years

fold_long <- results_df %>%
  pivot_longer(
    cols = starts_with("auc_"),
    names_to = "model_type",
    values_to = "auc"
  ) %>%
  mutate(model_label = case_when(
    model_type == "auc_glm" ~ "Logistic Regression (Weighted)",
    model_type == "auc_xgb" ~ "XGBoost (Weighted)",
    model_type == "auc_nn" ~ "Neural Network (Weighted)"
  ))

p1 <- ggplot(fold_long, aes(x = year, y = auc, color = model_label)) +
  geom_line(linewidth = 1.2, alpha = 0.8) +
  geom_point(size = 3) +
  scale_color_viridis_d(option = "D", end = 0.8) +
  scale_y_continuous(limits = c(0.5, 1.0), breaks = seq(0.5, 1.0, 0.05)) +
  labs(
    title = "ROC AUC: Expanding Window Validation",
    subtitle = "Higher is better (Balanced Evaluation)",
    x = "Validation Year",
    y = "ROC AUC",
    color = "Model"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

pr_long <- results_df %>%
  pivot_longer(
    cols = starts_with("pr_"),
    names_to = "model_type",
    values_to = "pr_auc"
  ) %>%
  mutate(model_label = case_when(
    model_type == "pr_glm" ~ "Logistic Regression (Weighted)",
    model_type == "pr_xgb" ~ "XGBoost (Weighted)",
    model_type == "pr_nn" ~ "Neural Network (Weighted)"
  ))

p2 <- ggplot(pr_long, aes(x = year, y = pr_auc, color = model_label)) +
  geom_line(linewidth = 1.2, alpha = 0.8) +
  geom_point(size = 3) +
  scale_color_viridis_d(option = "D", end = 0.8) +
  labs(
    title = "Precision-Recall AUC",
    subtitle = "Critical for Imbalanced Data (Rare Wildfires)",
    x = "Validation Year",
    y = "PR AUC",
    color = "Model"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

grid.arrange(p1, p2, ncol=2)

In [ ]:
results_text <- readLines("results.txt")

test_years <- as.numeric(str_extract(results_text[grep("Test", results_text)], "(?<=Test )\\d{4}"))

auc_lines <- results_text[grep("AUC ->", results_text)]
glm_aucs <- as.numeric(str_extract(auc_lines, "(?<=GLM: )\\d\\.\\d+"))
xgb_aucs <- as.numeric(str_extract(auc_lines, "(?<=XGB: )\\d\\.\\d+"))
nn_aucs  <- as.numeric(str_extract(auc_lines, "(?<=NN \\(Torch\\): )\\d\\.\\d+"))

parsed_results <- tibble(
  year = test_years,
  auc_glm = glm_aucs,
  auc_xgb = xgb_aucs,
  auc_nn = nn_aucs
)

plot_data <- parsed_results %>%
  pivot_longer(
    cols = starts_with("auc_"),
    names_to = "model",
    values_to = "auc"
  ) %>%
  mutate(model_label = case_when(
    model == "auc_glm" ~ "GLM (Baseline)",
    model == "auc_xgb" ~ "XGBoost (GPU)",
    model == "auc_nn" ~ "Neural Network"
  ))

ggplot(plot_data, aes(x = year, y = auc, color = model_label, group = model_label)) +
  geom_hline(yintercept = 0.5, linetype = "dashed", color = "gray50", alpha = 0.5) +

  geom_line(linewidth = 1.2, alpha = 0.8) +
  geom_point(size = 3, shape = 21, fill = "white", stroke = 2) +

  scale_color_manual(values = c(
    "GLM (Baseline)" = "#e74c3c",
    "XGBoost (GPU)" = "#2ecc71",
    "Neural Network" = "#3498db"
  )) +

  scale_y_continuous(
    limits = c(0.7, 1.0),
    breaks = seq(0.7, 1.0, 0.05),
    expand = c(0, 0)
  ) +
  scale_x_continuous(breaks = unique(plot_data$year)) +

  labs(
    title = "Model Performance Trajectory: Wildfire Prediction (1996-2005)",
    subtitle = "Area Under Curve (AUC) by Validation Year using Expanding Window Strategy",
    caption = "Data Source: Processed Grieb & Weather Data | Models: GLM, XGBoost, Torch NN",
    x = NULL,
    y = "AUC Score"
  ) +

  theme_minimal(base_size = 14, base_family = "sans") +
  theme(
    plot.background = element_rect(fill = "white", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_blank(),
    panel.grid.major.y = element_line(color = "gray90"),

    plot.title = element_text(face = "bold", size = 18, margin = margin(b = 10)),
    plot.subtitle = element_text(color = "gray40", size = 12, margin = margin(b = 20)),
    plot.caption = element_text(color = "gray60", size = 9, margin = margin(t = 20)),

    legend.position = "top",
    legend.title = element_blank(),
    legend.text = element_text(size = 11, face = "bold"),

    axis.text.x = element_text(angle = 45, hjust = 1, color = "gray30"),
    axis.text.y = element_text(color = "gray30")
  )

In [ ]:
results_content <- capture.output({
  cat("--- Wildfire Model Validation Results ---\n")
  cat("Timestamp: ", as.character(Sys.time()), "\n\n")

  print(results_df)

  cat("\n--- Aggregate Performance ---\n")
  cat("Mean GLM ROC-AUC: ", mean(results_df$auc_glm, na.rm=TRUE), "\n")
  cat("Mean XGB ROC-AUC: ", mean(results_df$auc_xgb, na.rm=TRUE), "\n")
  cat("Mean NN ROC-AUC: ", mean(results_df$auc_nn, na.rm=TRUE), "\n")

  cat("\nMean GLM PR-AUC: ", mean(results_df$pr_glm, na.rm=TRUE), "\n")
  cat("Mean XGB PR-AUC: ", mean(results_df$pr_xgb, na.rm=TRUE), "\n")
  cat("Mean NN PR-AUC: ", mean(results_df$pr_nn, na.rm=TRUE), "\n")
})

writeLines(results_content, "results.txt")
message("Results successfully saved to results.txt")

## 7. Deep Learning: Spatial CNN (Convolutional Neural Network)



In [ ]:
H_dim <- 32
W_dim <- 64

message(sprintf("CNN Input Setup: %d (Metric Height) x %d (Metric Width)", H_dim, W_dim))
message(sprintf("Total Input Channels: %d (One channel per input feature)", length(features_consolidated)))

model_df$y_idx <- as.numeric(cut(model_df$lat, breaks = H_dim, include.lowest = TRUE))
model_df$x_idx <- as.numeric(cut(model_df$lon, breaks = W_dim, include.lowest = TRUE))

time_steps <- model_df %>% select(year, month_date) %>% distinct() %>% arrange(year, month_date)
n_time <- nrow(time_steps)
n_feat <- length(features_consolidated)

X_tensor_all <- array(0, dim = c(n_time, n_feat, H_dim, W_dim))
Y_tensor_all <- array(0, dim = c(n_time, 1, H_dim, W_dim))
Mask_tensor_all <- array(0, dim = c(n_time, 1, H_dim, W_dim))

message("Filling tensors... (Mapping tabular data to grid)")

for(t in 1:n_time) {
  current_date <- time_steps$month_date[t]
  df_t <- model_df %>% filter(month_date == current_date)

  df_grid <- df_t %>%
    group_by(x_idx, y_idx) %>%
    summarise(
      across(all_of(features_consolidated), function(x) mean(x, na.rm=TRUE)),
      y_next_n = max(y_next_n, na.rm=TRUE),
      .groups="drop"
    )

  if(nrow(df_grid) > 0) {
    for(i in 1:nrow(df_grid)) {
      x <- df_grid$x_idx[i]
      y <- df_grid$y_idx[i]

      X_tensor_all[t, , y, x] <- as.numeric(df_grid[i, features_consolidated])
      Y_tensor_all[t, 1, y, x] <- df_grid$y_next_n[i]
      Mask_tensor_all[t, 1, y, x] <- 1
    }
  }
}

message("Data setup complete. Ready for CNN.")

### 7.2. Define CNN Architecture




In [ ]:
WildfireCNN <- nn_module(
  "WildfireCNN",
  initialize = function(in_channels) {

    self$conv_init <- nn_conv2d(in_channels, 64, kernel_size = 3, padding = 1)
    self$bn_init   <- nn_batch_norm2d(64)

    self$res1_conv1 <- nn_conv2d(64, 64, kernel_size = 3, padding = 1)
    self$res1_bn1   <- nn_batch_norm2d(64)
    self$res1_conv2 <- nn_conv2d(64, 64, kernel_size = 3, padding = 1)
    self$res1_bn2   <- nn_batch_norm2d(64)

    self$res2_conv1 <- nn_conv2d(64, 128, kernel_size = 3, padding = 2, dilation = 2)
    self$res2_bn1   <- nn_batch_norm2d(128)
    self$res2_conv2 <- nn_conv2d(128, 128, kernel_size = 3, padding = 2, dilation = 2)
    self$res2_bn2   <- nn_batch_norm2d(128)
    self$res2_shortcut <- nn_conv2d(64, 128, kernel_size = 1)

    self$res3_conv1 <- nn_conv2d(128, 256, kernel_size = 3, padding = 1)
    self$res3_bn1   <- nn_batch_norm2d(256)
    self$res3_conv2 <- nn_conv2d(256, 256, kernel_size = 3, padding = 1)
    self$res3_bn2   <- nn_batch_norm2d(256)
    self$res3_shortcut <- nn_conv2d(128, 256, kernel_size = 1)

    self$bottleneck <- nn_conv2d(256, 64, kernel_size = 1)
    self$output_map <- nn_conv2d(64, 1, kernel_size = 1)

    self$act <- nn_relu()
    self$dropout <- nn_dropout2d(0.4)
  },

  forward = function(x) {
    x <- self$act(self$bn_init(self$conv_init(x)))

    identity <- x
    out <- self$act(self$res1_bn1(self$res1_conv1(x)))
    out <- self$res1_bn2(self$res1_conv2(out))
    x <- self$act(out + identity)

    identity <- self$res2_shortcut(x)
    out <- self$act(self$res2_bn1(self$res2_conv1(x)))
    out <- self$res2_bn2(self$res2_conv2(out))
    x <- self$act(out + identity) %>% self$dropout()

    identity <- self$res3_shortcut(x)
    out <- self$act(self$res3_bn1(self$res3_conv1(x)))
    out <- self$res3_bn2(self$res3_conv2(out))
    x <- self$act(out + identity)

    x %>% self$bottleneck() %>% self$act() %>% self$output_map()
  }
)

message("Deep Residual CNN Architecture Defined.")

In [ ]:
n_epochs <- 100

cnn <- WildfireCNN(in_channels = n_feat)
cnn$to(device = device)
optimizer <- optim_adam(cnn$parameters, lr = 0.0005)

message("--- Starting Expanded CNN Training (100 Epochs) ---")
for(epoch in 1:n_epochs) {
  cnn$train()
  optimizer$zero_grad()
  preds <- cnn(x_train)
  loss <- masked_loss_fn(preds, y_train, mask_train)
  loss$backward()
  optimizer$step()

  if(epoch %% 10 == 0 || epoch == 1) {
    message(sprintf("Epoch %3d | Loss: %.4f", epoch, loss$item()))
  }
}

cnn$eval()
with_no_grad({
  test_logits <- cnn(x_test)
  test_probs <- torch_sigmoid(test_logits)
  r_probs <- as.numeric(test_probs$to(device="cpu"))
  r_truth <- as.numeric(y_test$to(device="cpu"))
  r_mask  <- as.logical(mask_test$to(device="cpu") == 1)
  flat_probs <- r_probs[r_mask]
  flat_truth <- r_truth[r_mask]
})

roc_obj <- roc(flat_truth, flat_probs, quiet = TRUE)
auc_score <- as.numeric(auc(roc_obj))

fg <- flat_probs[flat_truth == 1]
bg <- flat_probs[flat_truth == 0]
pr_score <- tryCatch(PRROC::pr.curve(scores.class0 = fg, scores.class1 = bg)$auc.integral, error = function(e) 0)

message(sprintf("Final Expanded CNN -> ROC AUC: %.3f | PR-AUC: %.3f", auc_score, pr_score))

cnn_results <- c(
  "\n--- Deep CNN Final Results (100 Epochs) ---",
  paste("ROC AUC:", round(auc_score, 4)),
  paste("PR AUC:", round(pr_score, 4))
)
write(cnn_results, file = "results.txt", append = TRUE)
message("CNN results appended to results.txt")